# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library. We use the Croissant schema to discover metadata and programmatically load record sets and fields referring to them by their unique `@id` identifiers.

### Dataset Source
The dataset source is provided via the following Croissant schema URL:

https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

We begin by loading dataset metadata and records using the `mlcroissant` library. The metadata reveals detailed information about the dataset, including its structure and available record sets. All entities will be referenced using their `@id` fields.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# The Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Let's review which record sets, fields, and columns are available in the dataset. All entities are referenced by their `@id`. This enables us to programmatically access data and ensure precision.

In [ ]:
# Print all record sets and their fields (@id)

record_sets = list(dataset.record_sets)
print("Available record sets and field @ids:")

for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}, name: {rs.get('name','')}")
    fields = rs.get('field', [])
    # If field is a dict (single), wrap in list
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        print(f"  Field @id: {field['@id']} (name: {field.get('name','')})")
    print("-----")

# Let's sample a record from each record set
for rs in record_sets:
    rs_id = rs['@id']
    print(f"Sample records from RecordSet @id: {rs_id}")
    try:
        for i, record in enumerate(dataset.records(record_set=rs_id)):
            print(record)
            if i>=1:
                break
    except Exception as e:
        print(f"Could not load records: {e}")
    print("=====")

## 3. Data Extraction

Now let's extract the main record sets into Pandas DataFrames for analysis. You can reference record sets and fields using their `@id` values identified above.

In [ ]:
# Extract data from all available record sets into DataFrames

dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for RecordSet @id: {record_set_id}, shape: {df.shape}")
        print("Columns:", df.columns.tolist())
        print(df.head(2))
        print("---")
    else:
        print(f"No records found for RecordSet @id: {record_set_id}")

## 4. Exploratory Data Analysis (EDA)

We will now explore and process the extracted data. For the purposes of this notebook, let's select one of the record sets with tabular clinical and pathological variables for EDA.

We identify numeric fields and demonstrate filtering, normalization, and grouping. This prepares the data for downstream modeling or visualization.

In [ ]:
# Choose a record set with tabular clinical records
# For example purposes, we select the first record set with data

selected_rs_id = None
for rs_id in record_set_ids:
    if rs_id in dataframes:
        selected_rs_id = rs_id
        break

df = dataframes[selected_rs_id]
print(f"Selected RecordSet @id: {selected_rs_id}")
print("Columns:", df.columns.tolist())

# Select a numeric field based on column types
numeric_field = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
        break

if numeric_field:
    threshold = df[numeric_field].median()  # Use median for demonstration
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records where '{numeric_field}' > {threshold}")
    print(filtered_df.head())

    # Normalize numeric field
    mean = filtered_df[numeric_field].mean()
    std = filtered_df[numeric_field].std()
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mean) / std
    print(f"Normalized '{numeric_field}' for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a categorical field if available
    group_field = None
    for col in df.columns:
        if pd.api.types.is_string_dtype(df[col]) and col != numeric_field:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped by '{group_field}':")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization

Let's visualize the distribution of the numeric field and its relationship with the chosen group field. This section uses Matplotlib and Seaborn for visualization.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field], kde=True)
    plt.title(f"Distribution of '{numeric_field}' in RecordSet @id: {selected_rs_id}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    if group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.xticks(rotation=45)
        plt.title(f"'{numeric_field}' by '{group_field}'")
        plt.ylabel(numeric_field)
        plt.xlabel(group_field)
        plt.show()

## 6. Conclusion

In this notebook, we:
- Loaded FAIR^2 dataset metadata and record sets using the Croissant schema and `mlcroissant` library.
- Explored available data structure via `@id` referencing.
- Extracted tabular clinical records for analysis.
- Performed basic EDA including filtering, normalization, and grouping.
- Visualized numeric distributions and relationships between clinical variables.

You can extend this workflow by referencing specific `@id` values of clinical variables, columns, and field entities for more targeted analyses, supporting transparent and reproducible research with FAIR data principles.

Feel free to customize this notebook by specifying particular record set or field `@id` values for deeper domain investigation.